In [ ]:
from pathlib import Path

from dataset.dataset_tables_dict import get_dataset_name
from dataset.query_gen_factory import get_placeholders_fn, get_query_gen
from tools.validate_tool.query_validator_class import QueryValidator
from tools.validate_tool.sf_list_gen import gen_sf
from utils.cli_config import build_run_config
from utils.gen_common import parse_query_ids

short_name = "basef1a-11bv32"
benchmark = "ceb"
query_ids = parse_query_ids(short_name, "basef", benchmark=benchmark)

args = build_run_config(
    benchmark=benchmark,
    conv_name=short_name,
    query_list=",".join(map(str, query_ids)),
    notify=False,
    disable_repo_sync=True,
    max_scale_factor=2,
    replay_cache=False,
    storage_plan_snapshot=None,
    keep_csv=True,
    disable_openai_tracing=True,
    disable_wandb=True,
    auto_u=False,
    auto_finish=False,
    is_bespoke_storage=False,
    replay=False,
)

workspace_path = Path("./output")
workspace_path.mkdir(exist_ok=True)

cache_path = Path(args.artifacts_dir) / "cache"

snapshotter = None

# prepare query gen
gen_query_fn = get_query_gen(args.benchmark)
gen_placeholders_fn = get_placeholders_fn(
    args.benchmark, cache_path / "placeholders_cache"
)
parquet_path = args.artifacts_dir + f"/{get_dataset_name(args.benchmark)}_parquet/"

query_list = [q.strip() for q in args.query_list.split(",")]


# assemble default sf values for the selected benchmark
verify_sf_list, max_scale_factor = gen_sf(args.benchmark)

query_validator = QueryValidator(
    benchmark=args.benchmark,
    gen_query_fn=gen_query_fn,
    sf_list=verify_sf_list + [max_scale_factor],
    parquet_path=parquet_path,
    wandb_pin_worker=True,
    all_query_ids=query_list,
    num_random_query_instantiations=10,
    query_cache_dir=cache_path / "query_cache",
    validate_cache_dir=cache_path / "validate_tool",
    workspace_path=workspace_path,
    git_snapshotter=snapshotter,
)

Gen and exec 1 queries for SF2: 100%|██████████| 16/16 [00:00<00:00, 459.40it/s]


In [3]:
# get query instantiations and convert to arg list
args_list, instantiations, num_queries = (
    query_validator._get_instantiations_and_convert_to_arg_list(
        scale_factor=0.25,
        query_id=["1a"],
    )
)

print(
    f"Len: {len(args_list)}, Instantiations: {len(instantiations)}, Num queries: {num_queries}"
)

Len: 10, Instantiations: 10, Num queries: 1


In [4]:
print(args_list[0])

1a "('16')" "('6')" "('France:1908', 'France:1909', 'Italy:1913', 'USA:1954', 'USA:January 1903')" "('4-Track Stereo', '70 mm 6-Track', 'Dolby Digital', 'Mono', 'Silent', 'Stereo')" "('episode', 'movie', 'tv movie', 'tv series', 'video movie')" "('costume designer', 'production designer')" "('m')" "1975" "1875"
